In [2]:
# =============================================================================
# EDA Master Script: Setup, Data Load, and All 8 Charts
# =============================================================================

import os
import re
import sys
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

# Setup paths and environment
sys.path.append(str(Path.cwd().parent))
load_dotenv(Path.cwd().parent / ".env")

# Visual styling
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Create output directory
EDA_OUTPUT_DIR = Path.cwd().parent / "data" / "eda_outputs"
EDA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Setup complete. Saving charts to:", EDA_OUTPUT_DIR)

# Load the CSV we just generated
# Force the path to your Desktop project folder
csv_path = r"C:\Users\adity\OneDrive\Desktop\real_estate_dashboard\data\cleaned_listings.csv"
df = pd.read_csv(csv_path, encoding="utf-8-sig")
print(f"Data Loaded! Shape: {df.shape}")

# Temporary column for human-readable crores
df["price_cr"] = df["price_inr"] / 10_000_000

# -----------------------------------------------------------------------------
# CHART 1: Price Distribution
# -----------------------------------------------------------------------------
print("\nGenerating Chart 1...")
fig, ax = plt.subplots(figsize=(12, 6))
sns.histplot(data=df.dropna(subset=["price_cr"]), x="price_cr", bins=50, kde=True, color="#2E86AB", alpha=0.7, ax=ax)
mean_price, median_price = df["price_cr"].mean(), df["price_cr"].median()
ax.axvline(mean_price, color="#E84855", linestyle="--", linewidth=2, label=f"Mean: ₹{mean_price:.2f} Cr")
ax.axvline(median_price, color="#F4A261", linestyle="-.", linewidth=2, label=f"Median: ₹{median_price:.2f} Cr")
ax.set_title("Property Price Distribution — Mumbai Region", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("Price (Crores ₹)", fontsize=13)
ax.set_ylabel("Number of Listings", fontsize=13)
ax.legend(fontsize=11)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"₹{x:.0f} Cr"))
plt.tight_layout()
plt.savefig(EDA_OUTPUT_DIR / "01_price_distribution.png", bbox_inches="tight", dpi=150)
plt.close()

# -----------------------------------------------------------------------------
# CHART 2: Average Price by BHK Type
# -----------------------------------------------------------------------------
print("Generating Chart 2...")
bhk_counts = df["bhk"].value_counts()
common_bhk = bhk_counts[bhk_counts >= 10].index
df_bhk = df[df["bhk"].isin(common_bhk)].copy()
bhk_avg = df_bhk.groupby("bhk")["price_cr"].agg(["mean", "median", "count"]).reset_index().sort_values("mean")

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(bhk_avg["bhk"], bhk_avg["mean"], color=sns.color_palette("muted", len(bhk_avg)), edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, bhk_avg["mean"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05, f"₹{val:.2f} Cr", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_title("Average Price by BHK Type", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("BHK Type", fontsize=13)
ax.set_ylabel("Average Price (Crores ₹)", fontsize=13)
ax.set_xticks(range(len(bhk_avg)))
ax.set_xticklabels([f"{row['bhk']}\n(n={int(row['count'])})" for _, row in bhk_avg.iterrows()], fontsize=11)
plt.tight_layout()
plt.savefig(EDA_OUTPUT_DIR / "02_avg_price_by_bhk.png", bbox_inches="tight", dpi=150)
plt.close()

# -----------------------------------------------------------------------------
# CHART 3: Avg Price/Sqft by Locality (Top 15)
# -----------------------------------------------------------------------------
print("Generating Chart 3...")
locality_counts = df["locality_clean"].value_counts()
valid_localities = locality_counts[locality_counts >= 10].index
locality_avg = df[df["locality_clean"].isin(valid_localities)].groupby("locality_clean")["price_per_sqft"].mean().sort_values(ascending=False).head(15).reset_index()

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(locality_avg["locality_clean"], locality_avg["price_per_sqft"], color=sns.color_palette("viridis", len(locality_avg)), edgecolor="white")
ax.invert_yaxis()
for bar, val in zip(bars, locality_avg["price_per_sqft"]):
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height() / 2, f"₹{val:,.0f}", ha="left", va="center", fontsize=10)
ax.set_title("Top 15 Localities by Avg. Price per Sqft", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("Average Price per Sqft (₹)", fontsize=13)
ax.set_ylabel("Locality", fontsize=13)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"₹{x:,.0f}"))
plt.tight_layout()
plt.savefig(EDA_OUTPUT_DIR / "03_ppsf_by_locality.png", bbox_inches="tight", dpi=150)
plt.close()

# -----------------------------------------------------------------------------
# CHART 4: Price vs Area Scatter
# -----------------------------------------------------------------------------
print("Generating Chart 4...")
plot_df = df[(df["area_sqft"].between(200, 5000)) & (df["price_cr"].between(0.1, 15)) & df["bhk"].notna()].dropna(subset=["area_sqft", "price_cr", "bhk"])
common_bhk = ["1 BHK", "2 BHK", "3 BHK", "4 BHK", "Studio", "1 RK"]
plot_df = plot_df[plot_df["bhk"].isin(common_bhk)]

fig, ax = plt.subplots(figsize=(13, 7))
palette = {"Studio": "#A8DADC", "1 RK": "#457B9D", "1 BHK": "#1D3557", "2 BHK": "#E63946", "3 BHK": "#F4A261", "4 BHK": "#2A9D8F"}

for bhk_type, group in plot_df.groupby("bhk"):
    ax.scatter(group["area_sqft"], group["price_cr"], label=bhk_type, color=palette.get(bhk_type, "#888888"), alpha=0.5, s=30, edgecolors="none")

from numpy.polynomial.polynomial import polyfit
valid = plot_df[["area_sqft", "price_cr"]].dropna()
if len(valid) > 1:
    coef = polyfit(valid["area_sqft"], valid["price_cr"], deg=1)
    x_line = np.linspace(valid["area_sqft"].min(), valid["area_sqft"].max(), 100)
    ax.plot(x_line, coef[1] * x_line + coef[0], color="black", linewidth=1.5, linestyle="--", label="Trend line")

ax.set_title("Price vs Area (colored by BHK type)", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("Area (sqft)", fontsize=13)
ax.set_ylabel("Price (Crores ₹)", fontsize=13)
ax.legend(title="BHK Type", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=10)
corr = valid["area_sqft"].corr(valid["price_cr"])
ax.text(0.02, 0.95, f"Pearson r = {corr:.3f}", transform=ax.transAxes, fontsize=11, color="black", bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
plt.tight_layout()
plt.savefig(EDA_OUTPUT_DIR / "04_price_vs_area_scatter.png", bbox_inches="tight", dpi=150)
plt.close()

# -----------------------------------------------------------------------------
# CHART 5: Furnishing Pie Chart
# -----------------------------------------------------------------------------
print("Generating Chart 5...")
furnish_counts = df["furnishing"].value_counts()
furnish_counts = furnish_counts[furnish_counts.index != "Unknown"]

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(furnish_counts.values, labels=furnish_counts.index, autopct="%1.1f%%", startangle=90, colors=["#2E86AB", "#A23B72", "#F18F01"], wedgeprops={"edgecolor": "white", "linewidth": 2}, pctdistance=0.75)
centre_circle = plt.Circle((0, 0), 0.50, fc="white")
ax.add_artist(centre_circle)
for autotext in autotexts: autotext.set_fontsize(12); autotext.set_fontweight("bold")
ax.set_title("Furnishing Status of Listings", fontsize=16, fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig(EDA_OUTPUT_DIR / "05_furnishing_distribution.png", bbox_inches="tight", dpi=150)
plt.close()

# -----------------------------------------------------------------------------
# CHART 6: Price by Floor Bucket
# -----------------------------------------------------------------------------
print("Generating Chart 6...")
def floor_bucket(floor_num):
    if pd.isna(floor_num): return "Unknown"
    floor_num = int(floor_num)
    if floor_num < 0: return "Basement"
    elif floor_num == 0: return "Ground"
    elif 1 <= floor_num <= 3: return "1–3"
    elif 4 <= floor_num <= 6: return "4–6"
    elif 7 <= floor_num <= 10: return "7–10"
    elif 11 <= floor_num <= 15: return "11–15"
    else: return "16+"

df["floor_bucket"] = df["total_floors"].apply(floor_bucket) # Using total_floors as extracted in Phase 2
floor_order = [f for f in ["Basement", "Ground", "1–3", "4–6", "7–10", "11–15", "16+", "Unknown"] if f in df["floor_bucket"].unique()]

fig, ax = plt.subplots(figsize=(13, 6))
sns.boxplot(data=df[df["price_cr"] < 10], x="floor_bucket", y="price_cr", order=floor_order, palette="coolwarm", width=0.5, fliersize=3, flierprops=dict(alpha=0.3), ax=ax)
ax.set_title("Price Distribution by Floor Bucket", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("Floor Group", fontsize=13)
ax.set_ylabel("Price (Crores ₹)", fontsize=13)
plt.tight_layout()
plt.savefig(EDA_OUTPUT_DIR / "06_price_by_floor.png", bbox_inches="tight", dpi=150)
plt.close()

# -----------------------------------------------------------------------------
# CHART 7: City Price Comparison
# -----------------------------------------------------------------------------
print("Generating Chart 7...")
city_stats = df.groupby("city").agg(avg_price_cr=("price_cr", "mean"), avg_ppsf=("price_per_sqft", "mean"), count=("price_inr", "count")).reset_index().sort_values("avg_price_cr", ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
colors = sns.color_palette("Set2", len(city_stats))

bars1 = ax1.bar(city_stats["city"], city_stats["avg_price_cr"], color=colors, edgecolor="white")
for bar, val, n in zip(bars1, city_stats["avg_price_cr"], city_stats["count"]):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"₹{val:.2f} Cr\n(n={n:,})", ha="center", va="bottom", fontsize=10)
ax1.set_title("Average Property Price by City", fontsize=14, fontweight="bold")
ax1.set_xlabel("City", fontsize=12)
ax1.set_ylabel("Average Price (Crores ₹)", fontsize=12)

bars2 = ax2.bar(city_stats["city"], city_stats["avg_ppsf"], color=colors, edgecolor="white")
for bar, val in zip(bars2, city_stats["avg_ppsf"]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, f"₹{val:,.0f}", ha="center", va="bottom", fontsize=10)
ax2.set_title("Average Price per Sqft by City", fontsize=14, fontweight="bold")
ax2.set_xlabel("City", fontsize=12)
ax2.set_ylabel("Avg. Price per Sqft (₹)", fontsize=12)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"₹{x:,.0f}"))

plt.suptitle("City-wise Price Comparison", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(EDA_OUTPUT_DIR / "07_city_price_comparison.png", bbox_inches="tight", dpi=150)
plt.close()

# -----------------------------------------------------------------------------
# CHART 8: Correlation Heatmap
# -----------------------------------------------------------------------------
print("Generating Chart 8...")
numeric_cols = ["price_inr", "area_sqft", "price_per_sqft", "total_floors"]
corr_df = df[numeric_cols].dropna()
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn", center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Correlation Heatmap — Numeric Features", fontsize=16, fontweight="bold", pad=15)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", fontsize=11)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=11)
plt.tight_layout()
plt.savefig(EDA_OUTPUT_DIR / "08_correlation_heatmap.png", bbox_inches="tight", dpi=150)
plt.close()

print("\n✅ ALL 8 CHARTS SUCCESSFULLY GENERATED AND SAVED!")

Setup complete. Saving charts to: C:\Users\adity\OneDrive\Desktop\data\eda_outputs
Data Loaded! Shape: (68993, 10)

Generating Chart 1...
Generating Chart 2...
Generating Chart 3...
Generating Chart 4...
Generating Chart 5...
Generating Chart 6...
Generating Chart 7...
Generating Chart 8...

✅ ALL 8 CHARTS SUCCESSFULLY GENERATED AND SAVED!
